# 🤟 BISINDO YOLOv11-nano + EigenCAM — Pipeline Server (VS Code / Local)

**Notebook ini khusus untuk eksekusi di Server GPU (VS Code Server / JupyterLab):**
1. Cek GPU RTX 5090 & PyTorch CUDA
2. Install dependencies pendukung
3. Konfigurasi Roboflow API key & download dataset BISINDO (A–Z)
4. Training YOLOv11-nano 100 Epoch murni dari Epoch 1
5. Evaluasi resmi Test set (mAP@50, mAP@50-95, Precision, Recall, Speed)
6. Visualisasi EigenCAM XAI (k=1 Canonical) & Preview Hasil

---
> **Catatan:** Notebook ini berjalan langsung di sistem file server (`results/`), tanpa ketergantungan ke Google Drive.

## STEP 1 — Cek Akselerasi GPU NVIDIA RTX 5090

In [ ]:
import subprocess, torch
print('=== STATUS GPU SYSTEM ===')
subprocess.run(['nvidia-smi'])
print('\n=== STATUS PYTORCH CUDA ===')
print(f'PyTorch Versi   : {torch.__version__}')
print(f'CUDA Tersedia   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Nama GPU        : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Total VRAM      : {vram:.2f} GB')


## STEP 2 — Install Dependencies (Ultralytics, Roboflow, dll.)

In [ ]:
# Install dependencies jika belum terpasang
%pip install ultralytics==8.4.123 roboflow==1.4.1 python-dotenv==1.2.3 matplotlib seaborn


## STEP 3 — Konfigurasi Roboflow API Key
Masukkan API Key Roboflow Anda di bawah ini, lalu jalankan cell ini.

In [ ]:
ROBOFLOW_API_KEY = 'YOUR_API_KEY_HERE'  # <-- Masukkan API Key Roboflow Anda di sini

with open('.env', 'w') as f:
    f.write(f'ROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')
print('[OK] Roboflow API key berhasil disimpan ke .env')


## STEP 4 — Download & Verifikasi Dataset BISINDO

In [ ]:
import os
# Download dataset jika belum ada
if not os.path.exists('datasets/bisindo-dataset-1/data.yaml'):
    !python -m src.data.download_dataset
else:
    print('[INFO] Dataset sudah terunduh di datasets/bisindo-dataset-1')

# Verifikasi jumlah gambar
dataset_dir = 'datasets/bisindo-dataset-1'
for split in ['train', 'valid', 'test']:
    p = f'{dataset_dir}/{split}/images'
    if os.path.exists(p):
        print(f'  {split:6}: {len(os.listdir(p))} gambar')
print('\n[OK] Dataset siap untuk training!')


## STEP 5 — Training YOLOv11-nano (100 Epoch Murni)
Training dimulai dari awal (Epoch 1 sampai 100) dan hasilnya disimpan di folder lokal `results/`.

In [ ]:
# Jalankan training 100 epoch bersih
!python -m src.training.train --config configs/train_config.yaml --epochs 100 --project results --name bisindo_yolo11n


## STEP 6 — Evaluasi Resmi Model pada Test Set
Menghitung mAP@50, mAP@50-95, Precision, Recall, dan Inference Speed (ms/FPS).

In [ ]:
BEST_PT = 'results/bisindo_yolo11n/weights/best.pt'

!python -m src.evaluation.evaluate \
    --weights {BEST_PT} \
    --split test \
    --output results/evaluation


In [ ]:
# Tampilkan ringkasan metrik
import json, os
metrics_path = 'results/evaluation/metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        m = json.load(f)
    print('=' * 45)
    print('HASIL EVALUASI RESMI — BISINDO YOLOv11-nano')
    print('=' * 45)
    print(f"mAP@50     : {m['map50']*100:.2f}%")
    print(f"mAP@50-95  : {m['map50_95']*100:.2f}%")
    print(f"Precision  : {m['precision']*100:.2f}%")
    print(f"Recall     : {m['recall']*100:.2f}%")
    speed_ms = m['inference_ms']
    fps = 1000.0 / speed_ms if speed_ms > 0 else 0
    print(f"Speed      : {speed_ms:.2f} ms/gambar (~{fps:.1f} FPS)")
    print('=' * 45)
else:
    print('[INFO] Jalankan cell evaluasi di atas terlebih dahulu.')


## STEP 7 — Visualisasi EigenCAM XAI (k=1 Canonical)
Menggunakan canonical Eigen-CAM (k=1 komponen utama) agar heatmap fokus murni ke area gestur tangan tanpa noise background/baju.

In [ ]:
BEST_PT = 'results/bisindo_yolo11n/weights/best.pt'
TEST_IMAGES = 'datasets/bisindo-dataset-1/test/images'
OUTPUT_DIR  = 'results/eigencam_k1'

!python -m src.xai.eigencam \
    --weights {BEST_PT} \
    --source  {TEST_IMAGES} \
    --output  {OUTPUT_DIR} \
    --n-components 1


## STEP 8 — Preview Sampel Visualisasi Acak (Multi-Alfabet)

In [ ]:
import glob, random, matplotlib.pyplot as plt, matplotlib.image as mpimg

OUTPUT_DIR = 'results/eigencam_k1'
all_cams = [f for f in glob.glob(f'{OUTPUT_DIR}/*.*') if not f.endswith('.json')]

if all_cams:
    sample_cams = random.sample(all_cams, min(6, len(all_cams)))
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for i, path in enumerate(sample_cams):
        img = mpimg.imread(path)
        axes[i].imshow(img)
        fname = path.replace('\\', '/').split('/')[-1]
        axes[i].set_title(fname[:35], fontsize=9)
        axes[i].axis('off')
    plt.suptitle('Sampel Acak Hasil EigenCAM XAI (k=1 Canonical)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('[INFO] Belum ada hasil EigenCAM. Jalankan Step 7 dulu!')
